<a href="https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Method choice and setup

import os
import sys
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git"
REPO_DIR = "/content/flyrank-ml-internship-starter"

# Clone only if the repository is missing.
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

csv_path = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Dataset not found: {csv_path}"
    )

df = pd.read_csv(csv_path)

print("Method: Random Forest classifier")
print("Dataset shape:", df.shape)
print("Target: trend_direction == 'down'")


Method: Random Forest classifier
Dataset shape: (30000, 44)
Target: trend_direction == 'down'


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Client-aware split

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

df = df.copy()

# Same target definition used by the starter pipeline.
df["is_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Remove rows without a usable client ID.
df["client_id"] = df["client_id"].fillna("unknown").astype(str)

clients = df["client_id"].unique()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(clients)

test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = df["client_id"].isin(test_clients)

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Split strategy: client holdout")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

# Confirm there is no client overlap.
overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("Client overlap:", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

print("Client leakage check: PASSED")
print("Train declining rate:", round(train_df["is_declining"].mean(), 3))
print("Test declining rate:", round(test_df["is_declining"].mean(), 3))


Split strategy: client holdout
Train rows: 27675
Test rows: 2325
Train clients: 26
Test clients: 6
Client overlap: 0
Client leakage check: PASSED
Train declining rate: 0.555
Test declining rate: 0.391


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Setup and load dataset

# 3. Train Random Forest and compare with the baseline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Features available before the target trend outcome.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

# Explicitly exclude fields that can encode the outcome.
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

numeric_features = [
    c for c in numeric_features
    if c in df.columns and c not in leakage_fields
]

categorical_features = [
    c for c in categorical_features
    if c in df.columns and c not in leakage_fields
]

# Build feature matrix.
X_numeric = df[numeric_features].apply(
    pd.to_numeric,
    errors="coerce"
).replace([np.inf, -np.inf], np.nan).fillna(0)

X_categorical = (
    df[categorical_features]
    .fillna("unknown")
    .astype(str)
)

X_categorical = pd.get_dummies(
    X_categorical,
    prefix=categorical_features,
    dtype=float
)

X = pd.concat(
    [
        X_numeric.reset_index(drop=True),
        X_categorical.reset_index(drop=True)
    ],
    axis=1
)

y = df["is_declining"].astype(int).reset_index(drop=True)

# Same client split indices.
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Random Forest.
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]


# Precision@K helper.
def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top["actual"].mean()


# -----------------------------
# Baseline score
# -----------------------------
# Recreate the Week-4 baseline logic without using the target.
def percentile_rank(series):
    return pd.to_numeric(
        series,
        errors="coerce"
    ).fillna(0).rank(
        method="average",
        pct=True
    )


def normalize(series):
    values = pd.to_numeric(
        series,
        errors="coerce"
    ).fillna(0)

    minimum = values.min()
    maximum = values.max()

    if maximum == minimum:
        return pd.Series(
            np.zeros(len(values)),
            index=values.index
        )

    return (values - minimum) / (maximum - minimum)


baseline_visibility = percentile_rank(
    np.log1p(df["impressions_90d"])
)

baseline_freshness = percentile_rank(
    df["days_since_last_update"]
)

baseline_position = (
    (1 - normalize(
        df["avg_position"].clip(
            lower=1,
            upper=50
        )
    ))
    * baseline_visibility
    * (df["avg_position"] > 0).astype(int)
)

baseline_depth = (
    (1 - percentile_rank(df["word_count"]))
    * baseline_visibility
)

baseline_score = (
    0.40 * baseline_visibility
    + 0.30 * baseline_freshness
    + 0.25 * baseline_position
    + 0.05 * baseline_depth
).clip(0, 1)

baseline_test_scores = baseline_score.iloc[test_idx].to_numpy()


# -----------------------------
# Metrics
# -----------------------------
results = []

for name, scores in [
    ("Week-4 baseline", baseline_test_scores),
    ("Random Forest", model_scores)
]:

    predictions = (scores >= 0.5).astype(int)

    results.append({
        "method": name,
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Precision@100": precision_at_k(y_test, scores, 100),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            scores
        ),
        "Average Precision": average_precision_score(
            y_test,
            scores
        )
    })

results_df = pd.DataFrame(results)

print("Model comparison:")
display(results_df.round(3))

# Main metric.
baseline_p50 = results_df.loc[
    results_df["method"] == "Week-4 baseline",
    "Precision@50"
].iloc[0]

model_p50 = results_df.loc[
    results_df["method"] == "Random Forest",
    "Precision@50"
].iloc[0]

print(
    f"\nBaseline Precision@50: {baseline_p50:.3f}"
)

print(
    f"Random Forest Precision@50: {model_p50:.3f}"
)

if baseline_p50 > 0:
    print(
        f"Relative lift: {model_p50 / baseline_p50:.2f}x"
    )

print("\nFeature count:", X.shape[1])


Model comparison:


,method,Precision@20,Precision@50,Precision@100,Precision,Recall,F1,ROC-AUC,Average Precision
0,Week-4 baseline,0.15,0.24,0.36,0.499,0.189,0.274,0.627,0.468
1,Random Forest,0.70,0.70,0.67,0.560,0.752,0.642,0.744,0.601



Baseline Precision@50: 0.240
Random Forest Precision@50: 0.700
Relative lift: 2.92x

Feature count: 62


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. Error analysis and feature interpretation

test_results = test_df[
    [
        "content_id",
        "client_id",
        "trend_direction",
        "is_declining"
    ]
].copy()

test_results["model_score"] = model_scores
test_results["predicted_decline"] = (
    test_results["model_score"] >= 0.5
).astype(int)

# False positives:
# Model says declining, actual label says not declining.
false_positives = test_results[
    (test_results["predicted_decline"] == 1)
    & (test_results["is_declining"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

# False negatives:
# Actual declining, model gives low probability.
false_negatives = test_results[
    (test_results["predicted_decline"] == 0)
    & (test_results["is_declining"] == 1)
].sort_values(
    "model_score",
    ascending=True
)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nHighest-confidence false positives:")
display(
    false_positives.head(10)
)

print("\nHighest-confidence false negatives:")
display(
    false_negatives.head(10)
)


# -----------------------------
# Feature importance
# -----------------------------
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop model features:")
display(
    importance.head(15)
)


# -----------------------------
# Leakage checks
# -----------------------------
used_columns = set(
    numeric_features + categorical_features
)

known_leakage = [
    field for field in leakage_fields
    if field in used_columns
]

print("\nKnown leakage fields used:")
print(known_leakage)

assert not known_leakage, (
    f"Leakage detected: {known_leakage}"
)

# Client split check.
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

assert not (
    train_clients & test_clients
), "Client overlap detected."

print("Leakage checks: PASSED")
print("No target/trend-window fields were used as model features.")
print("No client overlap between train and test.")


False positives: 538
False negatives: 225

Highest-confidence false positives:


,content_id,client_id,trend_direction,is_declining,model_score,predicted_decline
25913,content_331182ca4cae,client_f74efabef1,up,0,0.750342,1
21530,content_b15a8dbdf66f,client_f74efabef1,up,0,0.741841,1
23250,content_d2dffcc697a4,client_f74efabef1,stable,0,0.741372,1
23750,content_e55b8ab078b0,client_f74efabef1,stable,0,0.735385,1
29874,content_a1dd3f309e08,client_f74efabef1,up,0,0.734161,1
10155,content_643f585dc7f7,client_f74efabef1,up,0,0.727532,1
5966,content_f5013794ba57,client_f74efabef1,new,0,0.723960,1
22509,content_caa14adc7531,client_f74efabef1,new,0,0.721451,1
25376,content_ed3a7fd12cf8,client_f74efabef1,new,0,0.721026,1
4249,content_db1cd41b4b4f,client_f74efabef1,up,0,0.718920,1



Highest-confidence false negatives:


,content_id,client_id,trend_direction,is_declining,model_score,predicted_decline
3879,content_34b14c00f80c,client_d4735e3a26,down,1,0.093702,0
5770,content_28b4223f4e5f,client_98a3ab7c34,down,1,0.094615,0
27177,content_79ac977c6e0b,client_f74efabef1,down,1,0.146505,0
25838,content_cbc3b52a2ac1,client_98a3ab7c34,down,1,0.157757,0
12864,content_f1ef151d5e36,client_d4735e3a26,down,1,0.178241,0
22991,content_472ce7ae14c0,client_d4735e3a26,down,1,0.180287,0
12076,content_230de4c50860,client_d4735e3a26,down,1,0.189489,0
13659,content_4c437dd8c1ee,client_d4735e3a26,down,1,0.195166,0
5608,content_a55d958ec725,client_d4735e3a26,down,1,0.196858,0
23810,content_37804210415c,client_d4735e3a26,down,1,0.199049,0



Top model features:


,feature,importance
13,days_with_impressions,0.140560
5,impressions_90d,0.119063
19,avg_position,0.102159
15,content_age_days,0.071502
16,age_tier_order,0.035369
18,ctr,0.031347
3,word_count,0.030420
37,age_tier_365+,0.029205
21,scroll_rate,0.028101
6,clicks_90d,0.027849



Known leakage fields used:
[]
Leakage checks: PASSED
No target/trend-window fields were used as model features.
No client overlap between train and test.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.